# MIMIC-IV Preprocessing
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Hilina Fissha Woreta**

This notebook handles the data prep side of the project , taking raw MIMIC-IV hospital data and turning it into something a model can actually use. That means cleaning up messy columns, defining the outcome variable, pulling in relevant features, and doing the train/val/test split.

Nothing fancy, just trying to be careful about it. The cleaner this is, the more trustworthy the analysis downstream.

## Setup

Importing what we need and pointing to the MIMIC data on Kaggle. Setting the random seed here once so results are reproducible if someone reruns this.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

random_seed = 42

mimic_dir = "/kaggle/input/datasets/hilinafissha16/mimic-iv"

files = os.listdir(mimic_dir)
print("files found:")
for f in files:
    print(" -", f)

files found:
 - discharge.csv
 - d_labitems.csv
 - procedureevents.csv
 - d_icd_diagnoses.csv
 - patients.csv
 - transfers.csv
 - icustays.csv
 - diagnoses_icd.csv
 - admissions.csv
 - labevents.csv
 - d_items.csv


## Loading the raw tables

MIMIC-IV is spread across multiple CSVs. We only pull the columns we actually need since some of these files are massive (labevents alone has 158 million rows).

Quick rundown of what each table has:
- **admissions** , one row per hospital visit, includes dates, race, insurance
- **patients** , one row per patient, has age and sex
- **diagnoses_icd** , all diagnosis codes recorded per visit
- **icustays** , records for patients who went to the ICU
- **labevents** , every lab result, hence the 158M rows

In [2]:
admissions = pd.read_csv(
    f"{mimic_dir}/admissions.csv",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime",
             "deathtime", "admission_type", "insurance", "race",
             "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime", "deathtime"]
)

patients = pd.read_csv(
    f"{mimic_dir}/patients.csv",
    usecols=["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
    parse_dates=["dod"]
)

diagnoses = pd.read_csv(
    f"{mimic_dir}/diagnoses_icd.csv",
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"]
)

icustays = pd.read_csv(
    f"{mimic_dir}/icustays.csv",
    usecols=["subject_id", "hadm_id", "intime", "outtime", "los"]
)

labevents = pd.read_csv(
    f"{mimic_dir}/labevents.csv",
    usecols=["subject_id", "hadm_id", "itemid", "valuenum", "storetime"],
    parse_dates=["storetime"]
)

print("admissions:", len(admissions))
print("patients:", len(patients))
print("diagnoses:", len(diagnoses))
print("icustays:", len(icustays))
print("labevents:", len(labevents))

admissions: 546028
patients: 364627
diagnoses: 6364488
icustays: 94458
labevents: 158374764


## Building the base table

Merging admissions with patients to get demographics alongside each visit. MIMIC doesn't store age directly , it gives you an anchor year and age so you have to calculate it. We also compute length of stay in days.

Filtering to adults (18+) with a valid discharge time. Any pediatric cases or records with obvious data issues get dropped here.

In [3]:
# merge with patients to get demographics on each visit
base = admissions.merge(
    patients[["subject_id", "gender", "anchor_age", "anchor_year", "dod"]],
    on="subject_id",
    how="left"
)

# age at admission , MIMIC uses anchor year/age so we have to compute it
base["admit_year"] = base["admittime"].dt.year
base["age"] = base["anchor_age"] + (base["admit_year"] - base["anchor_year"])
base["age"] = base["age"].clip(18, 100)

# LOS in days
base["los_days"] = (base["dischtime"] - base["admittime"]).dt.total_seconds() / 86400
base["los_days"] = base["los_days"].clip(lower=0)

# drop pediatric cases and anything missing a discharge time
base = base[
    (base["age"] >= 18) &
    (base["dischtime"].notna()) &
    (base["los_days"] > 0)
].copy()

print("adult admissions after filtering:", len(base))
print("age range:", base["age"].min(), "-", base["age"].max())

adult admissions after filtering: 545848
age range: 18 - 100


## Cleaning the protected attributes

Race, insurance, and sex are central to this project, and they're stored as messy free text in MIMIC. There are a ton of variations for the same group , "BLACK/AFRICAN AMERICAN", "BLACK/AFRICAN", "BLACK/CAPE VERDEAN" all mean the same thing but show up as separate strings.

Mapping everything down to consistent categories. If we left the raw values in, the model would treat those as three different groups, which would completely break the fairness analysis.

In [4]:
# race is stored as messy free text in MIMIC , tons of variations for the same group
# collapsing everything down to 6 categories

race_map = {
    "WHITE": "White",
    "WHITE - BRAZILIAN": "White",
    "WHITE - EASTERN EUROPEAN": "White",
    "WHITE - OTHER EUROPEAN": "White",
    "WHITE - RUSSIAN": "White",
    "PORTUGUESE": "White",
    "BLACK/AFRICAN AMERICAN": "Black/African American",
    "BLACK/AFRICAN": "Black/African American",
    "BLACK/CAPE VERDEAN": "Black/African American",
    "BLACK/CARIBBEAN ISLAND": "Black/African American",
    "HISPANIC OR LATINO": "Hispanic/Latino",
    "HISPANIC/LATINO - CENTRAL AMERICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - COLOMBIAN": "Hispanic/Latino",
    "HISPANIC/LATINO - CUBAN": "Hispanic/Latino",
    "HISPANIC/LATINO - DOMINICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - GUATEMALAN": "Hispanic/Latino",
    "HISPANIC/LATINO - HONDURAN": "Hispanic/Latino",
    "HISPANIC/LATINO - MEXICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - PUERTO RICAN": "Hispanic/Latino",
    "HISPANIC/LATINO - SALVADORAN": "Hispanic/Latino",
    "SOUTH AMERICAN": "Hispanic/Latino",
    "ASIAN": "Asian",
    "ASIAN - ASIAN INDIAN": "Asian",
    "ASIAN - CHINESE": "Asian",
    "ASIAN - KOREAN": "Asian",
    "ASIAN - SOUTH EAST ASIAN": "Asian",
    "ASIAN - VIETNAMESE": "Asian",
    "AMERICAN INDIAN/ALASKA NATIVE": "Other/Unknown",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER": "Other/Unknown",
    "MULTIPLE RACE/ETHNICITY": "Other/Unknown",
    "OTHER": "Other/Unknown",
    "PATIENT DECLINED TO ANSWER": "Other/Unknown",
    "UNABLE TO OBTAIN": "Other/Unknown",
    "UNKNOWN": "Other/Unknown",
}

base["race_clean"] = (
    base["race"]
    .str.strip()
    .str.upper()
    .map(race_map)
    .fillna("Other/Unknown")
)

# same deal with insurance
def clean_insurance(val):
    if pd.isna(val):
        return "Other"
    v = str(val).strip().upper()
    if "MEDICARE" in v:
        return "Medicare"
    if "MEDICAID" in v:
        return "Medicaid"
    if "PRIVATE" in v or "BLUE CROSS" in v or "COMMERCIAL" in v:
        return "Private"
    if "SELF" in v:
        return "Self-Pay"
    return "Other"

base["insurance_clean"] = base["insurance"].apply(clean_insurance)

# standardize sex
base["sex"] = base["gender"].map({"M": "Male", "F": "Female"}).fillna("Unknown")

print("race distribution:")
print(base["race_clean"].value_counts())
print("\ninsurance distribution:")
print(base["insurance_clean"].value_counts())
print("\nsex distribution:")
print(base["sex"].value_counts())

race distribution:
race_clean
White                     362492
Black/African American     89037
Other/Unknown              42520
Hispanic/Latino            32056
Asian                      19743
Name: count, dtype: int64

insurance distribution:
insurance_clean
Medicare    244500
Private     173360
Medicaid    104196
Other        23792
Name: count, dtype: int64

sex distribution:
sex
Female    284010
Male      261838
Name: count, dtype: int64


## Creating the readmission label

The outcome variable: did this patient come back within 30 days of discharge?

30 days is the standard clinical threshold , US hospitals get financially penalized for 30-day readmissions because it often means something went wrong (discharged too early, not enough follow-up, etc.). So it's not an arbitrary cutoff.

One edge case: patients who died in-hospital can't be readmitted, so we exclude them from the positive label.

In [5]:
# 30-day readmission label
# for each admission, did this patient come back within 30 days of discharge?

base = base.sort_values(["subject_id", "admittime"]).reset_index(drop=True)

# next admission time per patient
base["next_admittime"] = base.groupby("subject_id")["admittime"].shift(-1)

base["days_to_readmit"] = (
    base["next_admittime"] - base["dischtime"]
).dt.total_seconds() / 86400

# positive only if they came back within 30 days AND didn't die this admission
base["readmitted_30d"] = (
    (base["days_to_readmit"] >= 0) &
    (base["days_to_readmit"] <= 30) &
    (base["hospital_expire_flag"] == 0)
).astype(int)

base["readmitted_30d"] = base["readmitted_30d"].fillna(0).astype(int)

print("readmission rate:", round(base["readmitted_30d"].mean() * 100, 1), "%")
print("readmitted:", base["readmitted_30d"].sum())
print("not readmitted:", (base["readmitted_30d"] == 0).sum())

readmission rate: 20.0 %
readmitted: 109291
not readmitted: 436557


## Comorbidity flags

Each admission has a list of ICD-10 codes. We use those to create 16 binary flags , one per condition. So `cm_diabetes = 1` means diabetes was documented during that stay.

These are conditions known to be associated with readmission risk: heart failure, kidney disease, depression, substance use disorders, etc. We also add a simple count of how many a patient has.

In [6]:
# binary flag per comorbidity using ICD-10 codes
# 1 = condition documented during this admission, 0 = not

comorbidity_map = {
    "chf":           ["I50"],
    "arrhythmia":    ["I44", "I45", "I46", "I47", "I48", "I49"],
    "hypertension":  ["I10", "I11", "I12", "I13", "I15"],
    "cpd":           ["J40", "J41", "J42", "J43", "J44", "J45", "J46", "J47"],
    "diabetes":      ["E10", "E11", "E12", "E13", "E14"],
    "renal_failure": ["N17", "N18", "N19"],
    "liver_disease": ["K70", "K71", "K72", "K73", "K74"],
    "cancer":        ["C00", "C01", "C02", "C03", "C04", "C05", "C06",
                      "C07", "C08", "C09", "C10", "C11", "C12", "C13",
                      "C14", "C15", "C16", "C17", "C18", "C19", "C20"],
    "obesity":       ["E66"],
    "depression":    ["F32", "F33"],
    "anxiety":       ["F40", "F41"],
    "alcohol_abuse": ["F10"],
    "drug_abuse":    ["F11", "F12", "F13", "F14", "F15", "F16", "F18", "F19"],
    "psychosis":     ["F20", "F22", "F23", "F24", "F25"],
    "coagulopathy":  ["D65", "D66", "D67", "D68", "D69"],
    "aids":          ["B20", "B21", "B22", "B24"],
}

# ICD-10 only
icd10 = diagnoses[diagnoses["icd_version"] == 10].copy()
icd10["icd_code"] = icd10["icd_code"].str.strip().str.upper()

comorbidity_df = icd10[["hadm_id"]].drop_duplicates()

for flag, prefixes in comorbidity_map.items():
    pattern = "|".join([f"^{p}" for p in prefixes])
    matched = icd10[icd10["icd_code"].str.match(pattern, na=False)]["hadm_id"].unique()
    comorbidity_df[f"cm_{flag}"] = comorbidity_df["hadm_id"].isin(matched).astype(int)

# total condition count per admission
cm_cols = [c for c in comorbidity_df.columns if c.startswith("cm_")]
comorbidity_df["comorbidity_count"] = comorbidity_df[cm_cols].sum(axis=1)

print("comorbidity flags created:", len(cm_cols))
print("\nprevalence of each condition:")
print(comorbidity_df[cm_cols].mean().round(3).sort_values(ascending=False))

comorbidity flags created: 16

prevalence of each condition:
cm_hypertension     0.536
cm_renal_failure    0.261
cm_diabetes         0.258
cm_arrhythmia       0.219
cm_depression       0.194
cm_cpd              0.186
cm_anxiety          0.171
cm_chf              0.169
cm_obesity          0.122
cm_coagulopathy     0.110
cm_alcohol_abuse    0.078
cm_liver_disease    0.062
cm_drug_abuse       0.060
cm_psychosis        0.025
cm_cancer           0.016
cm_aids             0.006
dtype: float64


## Lab features

Lab values are one of the best signals for how sick a patient actually is at discharge. We pick 9 standard tests , creatinine, hemoglobin, sodium, potassium, etc.

Since patients get labs drawn multiple times during a stay, we use the last value before discharge. That's the most recent picture of the patient right before they leave, which is what matters for predicting what happens next.

In [7]:
# 9 clinically relevant lab tests , using last value before discharge
lab_items = {
    50912: "creatinine",
    51006: "bun",
    51222: "hemoglobin",
    51301: "wbc",
    50931: "glucose",
    50971: "potassium",
    50983: "sodium",
    51265: "platelets",
    50882: "bicarbonate",
}

labs = labevents[labevents["itemid"].isin(lab_items.keys())].copy()

# attach discharge time so we can filter to pre-discharge labs only
labs = labs.merge(
    base[["hadm_id", "dischtime"]],
    on="hadm_id",
    how="inner"
)

labs = labs[labs["storetime"] <= labs["dischtime"]]

# last recorded value per lab per admission
labs = (
    labs.sort_values("storetime")
    .groupby(["hadm_id", "itemid"])["valuenum"]
    .last()
    .reset_index()
)

labs["lab_name"] = labs["itemid"].map(lab_items)

# pivot to wide format
lab_features = labs.pivot(
    index="hadm_id",
    columns="lab_name",
    values="valuenum"
).reset_index()
lab_features.columns.name = None

print("lab features shape:", lab_features.shape)
print("\nmissing rate per lab:")
print(lab_features.drop(columns="hadm_id").isna().mean().round(3).sort_values(ascending=False))

lab features shape: (432555, 10)

missing rate per lab:
glucose        0.062
bicarbonate    0.060
sodium         0.053
potassium      0.049
bun            0.049
creatinine     0.040
wbc            0.026
hemoglobin     0.024
platelets      0.020
dtype: float64


### Who has missing lab results?

Before imputing, worth checking whether the missingness is actually random or concentrated in certain groups. This matters for the fairness analysis , if minority patients are disproportionately missing labs, that's both a real-world inequality and a methodological problem.

If we just dropped patients with missing labs, we'd be quietly removing a lot of minority patients from the dataset.

In [8]:
# checking whether missing labs are random or skewed toward certain groups
# temporary merge just for this check , not kept

temp = base[["hadm_id", "race_clean", "insurance_clean"]].merge(
    lab_features, on="hadm_id", how="left"
)

temp["any_lab_missing"] = temp[["bicarbonate", "bun", "creatinine",
                                 "glucose", "hemoglobin", "platelets",
                                 "potassium", "sodium", "wbc"]].isna().any(axis=1)

missing_group = temp[temp["any_lab_missing"] == True]
complete_group = temp[temp["any_lab_missing"] == False]

print("patients with at least one missing lab:", len(missing_group))
print("patients with complete labs:", len(complete_group))

print("\nrace distribution - missing labs (%):")
print((missing_group["race_clean"].value_counts() / len(missing_group) * 100).round(1))

print("\nrace distribution - complete labs (%):")
print((complete_group["race_clean"].value_counts() / len(complete_group) * 100).round(1))

print("\ninsurance distribution - missing labs (%):")
print((missing_group["insurance_clean"].value_counts() / len(missing_group) * 100).round(1))

print("\ninsurance distribution - complete labs (%):")
print((complete_group["insurance_clean"].value_counts() / len(complete_group) * 100).round(1))

del temp

patients with at least one missing lab: 146479
patients with complete labs: 399369

race distribution - missing labs (%):
race_clean
White                     59.7
Black/African American    20.5
Hispanic/Latino            7.7
Other/Unknown              7.6
Asian                      4.5
Name: count, dtype: float64

race distribution - complete labs (%):
race_clean
White                     68.9
Black/African American    14.8
Other/Unknown              7.9
Hispanic/Latino            5.2
Asian                      3.3
Name: count, dtype: float64

insurance distribution - missing labs (%):
insurance_clean
Private     39.7
Medicare    27.5
Medicaid    25.9
Other        6.9
Name: count, dtype: float64

insurance distribution - complete labs (%):
insurance_clean
Medicare    51.1
Private     28.8
Medicaid    16.6
Other        3.4
Name: count, dtype: float64


## Dataset Overview and Key Findings

The final dataset has **545,848 adult admissions** from **364,627 unique patients** , so many patients have multiple visits. White patients are ~66% of admissions, Black/African American ~16%. Medicare is the most common insurance type. The 20% readmission rate is in line with what gets reported nationally. Hypertension (54%) is by far the most common comorbidity.

**On missing labs:** missingness is not random. Black and Hispanic patients, and those on Medicaid, are overrepresented in the missing-lab group, while White and Medicare patients are underrepresented. That's a real signal about unequal access to testing, not just a data artifact , which is why we impute instead of drop.

## ICU stay features

Only about 15% of patients end up in the ICU. For those who do, total ICU time and number of ICU stays per admission are both useful signals. Multiple ICU stays in one admission tells you something different than a single routine visit.

In [9]:
# for patients with ICU time, sum up total hours and count number of stays
# some patients bounce in and out of ICU multiple times in one admission

icu_features = icustays.groupby("hadm_id").agg(
    icu_los_total=("los", "sum"),
    n_icu_stays=("los", "count")
).reset_index()

print("admissions with icu stay:", len(icu_features))
print("average icu length of stay (days):", round(icu_features["icu_los_total"].mean(), 2))
print("patients with more than one icu stay:", (icu_features["n_icu_stays"] > 1).sum())

admissions with icu stay: 85242
average icu length of stay (days): 4.02
patients with more than one icu stay: 7694


## Prior admissions in the last 12 months

One of the stronger predictors of readmission is just how many times the patient has already been admitted in the past year. Someone coming in for the fifth time is in a very different situation than a first-timer.

We do a self-join on the admissions table to count prior visits within a 365-day window.

In [10]:
# count how many times each patient was admitted in the 12 months before each visit
# self-join on subject_id to compare each admission against earlier ones

base_sorted = base[["subject_id", "hadm_id", "admittime"]].sort_values(
    ["subject_id", "admittime"]
)

prior = base_sorted.merge(
    base_sorted.rename(columns={
        "hadm_id": "prev_hadm_id",
        "admittime": "prev_admittime"
    }),
    on="subject_id",
    how="left"
)

# only keep prior admissions within the 12-month window
prior = prior[
    (prior["prev_admittime"] < prior["admittime"]) &
    (prior["prev_admittime"] >= prior["admittime"] - pd.Timedelta(days=365))
]

prior_counts = prior.groupby("hadm_id")["prev_hadm_id"].count().reset_index()
prior_counts.columns = ["hadm_id", "prior_admissions_12m"]

print("admissions with at least one prior visit:", len(prior_counts))
print("average prior admissions:", round(prior_counts["prior_admissions_12m"].mean(), 2))
print("max prior admissions:", prior_counts["prior_admissions_12m"].max())

admissions with at least one prior visit: 236254
average prior admissions: 2.86
max prior admissions: 69


## Assembling the full feature table

Putting everything together , demographics, comorbidities, lab values, ICU features, prior admissions , into one table, one row per admission.

Also adding two intersectional columns: `race_x_sex` and `race_x_insurance`. These let us check for bias that only shows up at the intersection, like whether Black women on Medicaid specifically are disadvantaged in ways that looking at race or insurance alone wouldn't reveal.

In [11]:
# merge everything into one table: demographics, comorbidities, labs, ICU, prior admissions

features = (
    base[[
        "subject_id", "hadm_id", "admittime", "dischtime",
        "age", "sex", "race_clean", "insurance_clean",
        "admission_type", "los_days", "hospital_expire_flag",
        "readmitted_30d"
    ]]
    .merge(comorbidity_df, on="hadm_id", how="left")
    .merge(lab_features,   on="hadm_id", how="left")
    .merge(icu_features,   on="hadm_id", how="left")
    .merge(prior_counts,   on="hadm_id", how="left")
)

# fill nulls from left joins
for col in cm_cols:
    features[col] = features[col].fillna(0).astype(int)

features["comorbidity_count"]    = features["comorbidity_count"].fillna(0).astype(int)
features["n_icu_stays"]          = features["n_icu_stays"].fillna(0).astype(int)
features["icu_los_total"]        = features["icu_los_total"].fillna(0)
features["prior_admissions_12m"] = features["prior_admissions_12m"].fillna(0).astype(int)

# intersectional columns for the fairness analysis
features["race_x_sex"]       = features["race_clean"] + "_" + features["sex"]
features["race_x_insurance"] = features["race_clean"] + "_" + features["insurance_clean"]

print("final feature table shape:", features.shape)
print("\ncolumns:", list(features.columns))
print("\nmissing values per column:")
missing = features.isna().mean()
print(missing[missing > 0].round(3))

final feature table shape: (545848, 43)

columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'age', 'sex', 'race_clean', 'insurance_clean', 'admission_type', 'los_days', 'hospital_expire_flag', 'readmitted_30d', 'cm_chf', 'cm_arrhythmia', 'cm_hypertension', 'cm_cpd', 'cm_diabetes', 'cm_renal_failure', 'cm_liver_disease', 'cm_cancer', 'cm_obesity', 'cm_depression', 'cm_anxiety', 'cm_alcohol_abuse', 'cm_drug_abuse', 'cm_psychosis', 'cm_coagulopathy', 'cm_aids', 'comorbidity_count', 'bicarbonate', 'bun', 'creatinine', 'glucose', 'hemoglobin', 'platelets', 'potassium', 'sodium', 'wbc', 'icu_los_total', 'n_icu_stays', 'prior_admissions_12m', 'race_x_sex', 'race_x_insurance']

missing values per column:
bicarbonate    0.255
bun            0.246
creatinine     0.239
glucose        0.256
hemoglobin     0.227
platelets      0.224
potassium      0.247
sodium         0.250
wbc            0.228
dtype: float64


## Handling missing lab values

Since we confirmed the missingness isn't random, we impute per race group rather than using the overall median. The overall median is dominated by White patients (they're ~66% of the dataset), so using it would effectively inject majority-group values into minority patient records.

We also add a binary indicator column for each lab before imputing, so the model can tell the difference between a real measurement and an imputed one.

In [12]:
# impute per race group instead of overall median
# overall median is dominated by White patients (~66% of data),
# so using it would silently push majority-group values onto minority records

lab_cols = ["bicarbonate", "bun", "creatinine", "glucose",
            "hemoglobin", "platelets", "potassium", "sodium", "wbc"]

# flag missingness before filling , model should know what was originally observed
for col in lab_cols:
    features[f"{col}_missing"] = features[col].isna().astype(int)

# per-race median, fall back to overall if a group has no data
for col in lab_cols:
    group_median = features.groupby("race_clean")[col].transform("median")
    overall_median = features[col].median()
    features[col] = features[col].fillna(group_median).fillna(overall_median)

print("missing values after imputation:")
print(features[lab_cols].isna().sum())
print("\nfinal shape:", features.shape)

missing values after imputation:
bicarbonate    0
bun            0
creatinine     0
glucose        0
hemoglobin     0
platelets      0
potassium      0
sodium         0
wbc            0
dtype: int64

final shape: (545848, 52)


## Train / validation / test split

70% train, 15% val, 15% test, stratified by both readmission label and race. Without stratification, we could end up with a test set accidentally skewed toward one group, which would make the fairness evaluation unreliable.

In [13]:
# 70/15/15 split, stratified by readmission label AND race
# keeps each split's demographic distribution consistent

features["strat_key"] = features["readmitted_30d"].astype(str) + "_" + features["race_clean"]

# drop strat groups too small to split
strat_counts = features["strat_key"].value_counts()
valid = strat_counts[strat_counts >= 3].index
rare  = strat_counts[strat_counts < 3].index

main = features[features["strat_key"].isin(valid)].copy()
leftover = features[features["strat_key"].isin(rare)].copy()

# train vs (val + test)
train, temp = train_test_split(
    main,
    test_size=0.30,
    stratify=main["strat_key"],
    random_state=random_seed
)

# val vs test
val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp["strat_key"],
    random_state=random_seed
)

# rare groups go into train
train = pd.concat([train, leftover], ignore_index=True)

for df in [train, val, test, features]:
    df.drop(columns=["strat_key"], inplace=True, errors="ignore")

print("train:", len(train), "| readmit rate:", round(train["readmitted_30d"].mean() * 100, 1), "%")
print("val:  ", len(val),   "| readmit rate:", round(val["readmitted_30d"].mean() * 100, 1), "%")
print("test: ", len(test),  "| readmit rate:", round(test["readmitted_30d"].mean() * 100, 1), "%")

train: 382093 | readmit rate: 20.0 %
val:   81877 | readmit rate: 20.0 %
test:  81878 | readmit rate: 20.0 %


## Saving the outputs

Saving everything as Parquet. Much faster and more compact than CSV at this scale. These files go straight into Stage 2 (the XGBoost model).

In [14]:
# saving as parquet , much faster and smaller than CSV at this scale

import os
os.makedirs("/kaggle/working/splits", exist_ok=True)

train.to_parquet("/kaggle/working/splits/train.parquet", index=False)
val.to_parquet("/kaggle/working/splits/val.parquet",     index=False)
test.to_parquet("/kaggle/working/splits/test.parquet",   index=False)

features.to_parquet("/kaggle/working/mimic_features.parquet", index=False)

print("files saved:")
print("  train.parquet  -", len(train), "rows")
print("  val.parquet    -", len(val), "rows")
print("  test.parquet   -", len(test), "rows")
print("  mimic_features.parquet -", len(features), "rows")

files saved:
  train.parquet  - 382093 rows
  val.parquet    - 81877 rows
  test.parquet   - 81878 rows
  mimic_features.parquet - 545848 rows
